In [ ]:
import numpy as np
from scipy.linalg import sqrtm, dft
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator


# ── Matrix utilities ───────────────────────────────────────────────────────────



x_lb = 0 
x_rb = 1
d = 2
n = 4
N_x =2**n

                           # grid points per dimension
A   = np.eye(2)                          # PDE coefficient matrix (replace as needed)
L = x_rb - x_lb
dx = L/N_x
y_lb = x_lb
y_ub = x_rb
N_y = N_x
xs, ys = np.meshgrid(np.linspace(x_lb, x_rb, N_x, endpoint=False), 
                    np.linspace(y_lb, y_ub, N_y, endpoint=False), 
                    indexing='ij')

def f(x, y) : 
    return np.cos(2*np.pi *x) * np.sin(-4*np.pi * y )
    

    
"""Constructs the 1D DFT matrix of size N."""
dfmtx = np.fft.fft(np.eye(N_x))#/np.sqrt(N_x)

"""Constructs the 2D DFT matrix of size N x N as a Kronecker product."""
FG = np.kron(dfmtx, dfmtx)
GF =  np.kron(
    (np.conj(dfmtx).T)/N_x, 
    (np.conj(dfmtx).T)/N_x, 
)
def solver_Elliptic(f): 
    f_values = f(xs, ys)

    f_h = np.fft.fft2(f_values)
    wave = np.fft.fftfreq(N_x, d= dx) * 2j* np.pi
    wave[0] = 1
    wave_real = np.fft.fftfreq(N_y, d= dx)* 2j* np.pi
    wave_real[0] = 1
    k_x, k_y = np.meshgrid(wave, wave_real, indexing='ij')
    

    u_h = f_h/ (A[0,0]*k_x**2 + + A[1, 0]* k_x * k_y + A[0, 1] * k_y *k_x + A[1,1]*k_y**2)

    return np.fft.ifft2(u_h, s= (N_x, N_y))
f_vec   = f(xs, ys)
f_vec   = f_vec.flatten()
def solver_Elliptic_FG(f): 

    # Fourier transform
    f_flatten = f(xs, ys).flatten()
    f_h = FG @ f_flatten

    # Build laplacian in spectral domain
    D = np.diag( 
        spectral_eigenvalues(N_x)
    )
    

    Elliptic_spec = A[0,0]*np.kron(D**2, np.eye(N_x)) + A[0, 1] * np.kron(D, D) + A[1, 0] * np.kron(D, D) + A[1, 1]*np.kron(np.eye(N_x), D**2)

    Elliptic_spec[0, 0] = 1 # avoid zero inversion
    inverse_Elliptic = np.linalg.inv(Elliptic_spec)


    # Apply Laplacian in spectral domain and Inverse FFT
    u_flatten = GF @ inverse_Elliptic @ f_h

    return u_flatten.reshape(N_x, N_x)

# ── Build operators ────────────────────────────────────────────────────────
print("Building inverse elliptic operator...")
inverse_Elliptic = build_elliptic_inverse(N_x, A)

# ── Classical solution ─────────────────────────────────────────────────────
print("\nRunning classical solver...")
u_classical = solver_Elliptic_FG(f)
u_exact = solver_Elliptic(f)
error_classical = np.linalg.norm(u_classical - u_exact) / np.linalg.norm(u_exact)
print(f"  Classical relative error: {error_classical:.6e}")

# ── Quantum circuit ────────────────────────────────────────────────────────
print("\nBuilding quantum circuit...")
qc, alpha = build_elliptic_circuit(inverse_Elliptic, n)
mat       = extract_unitary(qc)

verify_encoding(mat, inverse_Elliptic, alpha, n)

# ── Quantum solution ───────────────────────────────────────────────────────
print("\nComputing quantum solution...")
block_size =  2**(d*n)
leading_block = mat[:block_size, :block_size]
print(f"  Full unitary shape: {mat.shape}")
print(f"  Leading block shape: {leading_block.shape}")
f_norm        = np.linalg.norm(f_vec)
f_normalized  = f_vec / f_norm
print(f"  Shape of normalized f: {f_normalized.shape}")
u_quantum = leading_block @ (alpha * f_normalized) * f_norm
u_quantum = u_quantum.reshape(N_x, N_x)
error_quantum = np.linalg.norm(u_quantum - u_exact) / np.linalg.norm(u_exact)
print(f"  Quantum relative error:   {error_quantum:.6e}")

print(f"\n{'─'*45}")
print(f"  Classical error : {error_classical:.6e}")
print(f"  Quantum error   : {error_quantum:.6e}")

Building inverse elliptic operator...
  Condition number of elliptic op: 1.277e+07

Running classical solver...
  Classical relative error: 1.007450e+00

Building quantum circuit...
  Building block encoding (make_unitary)...
  Unitary shape: (512, 512)
  Composing circuit...
  Simulating unitary...


C:\Users\giaco\AppData\Local\Temp\ipykernel_12992\325770228.py:82: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qft  = QFT(n)
C:\Users\giaco\AppData\Local\Temp\ipykernel_12992\325770228.py:83: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qft1 = QFT(n)


  Block-encoding error: 1.147665e-15

Computing quantum solution...
  Full unitary shape: (512, 512)
  Leading block shape: (256, 256)
  Shape of normalized f: (256,)
  Quantum relative error:   1.007450e+00

─────────────────────────────────────────────
  Classical error : 1.007450e+00
  Quantum error   : 1.007450e+00
